# Microsoft 365 Audit Log Analysis — Per-User PDF Reports

This notebook processes Microsoft 365 audit log CSV exports (from audit.microsoft.com) and generates **one PDF report per user**. Each report is designed to help HR evaluate whether a departing user engaged in suspicious activity before leaving the company.

**Reports include:**
- Operation type counts (delete, download, modify, share, etc.)
- Per-site breakdown of operations
- Daily activity timeline with spike detection
- Off-hours activity analysis
- Risk indicators with severity levels
- Comparison of last-week vs. monthly-average activity
- Top files involved in sensitive operations (delete/download/share)

In [ ]:
# Install dependencies in the venv
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "matplotlib", "seaborn", "fpdf2", "openpyxl"])

## Check Input Files

In [ ]:
import os, glob
# Quick sanity check — list input CSV files
csv_files = glob.glob(os.path.join("input", "*.csv"))
print(f"Found {len(csv_files)} CSV file(s) in input/:")
for f in csv_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f)}  ({size_mb:.1f} MB)")

## Run Report Generation

The `generate_reports.py` script:
1. Loads all CSVs from `input/`
2. Parses the AuditData JSON to extract sites, files, IPs, platforms, etc.
3. Categorises operations (Delete, Download, Share, Modify, Upload, etc.)
4. For each user: analyses risk indicators, creates charts, and generates a multi-page PDF
5. Saves reports to `output/`

In [ ]:
# Run the report generation script (contains all analysis + PDF generation logic)
# The script reads CSVs from input/, analyses per-user activity, and writes PDFs to output/
%run generate_reports.py

## Validate Generated Reports

In [ ]:
# Validate generated outputs
import fitz  # PyMuPDF — install with: pip install pymupdf

print("Generated files in output/:\n")
for f in sorted(glob.glob(os.path.join("output", "*.*"))):
    size_kb = os.path.getsize(f) / 1024
    ext = os.path.splitext(f)[1]
    label = "PDF report" if ext == ".pdf" else "Excel records"
    print(f"  [{label}]  {os.path.basename(f)}  ({size_kb:.1f} KB)")
print()

# Validate PDFs
pdf_files = sorted(glob.glob(os.path.join("output", "*.pdf")))
for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    all_text = "".join(page.get_text() for page in doc)
    lines = [l.strip() for l in all_text.split('\n') if l.strip()]
    key_lines = [l for l in lines if any(kw in l for kw in
                 ['Risk Score', 'Total operations', '[HIGH]', '[MEDIUM]', '[LOW]'])]
    print(f"  {os.path.basename(pdf_path)}: {len(doc)} pages")
    for l in key_lines:
        print(f"    {l}")
    doc.close()
    print()

# Validate Excel files
xlsx_files = sorted(glob.glob(os.path.join("output", "*.xlsx")))
import pandas as pd
for xlsx_path in xlsx_files:
    df = pd.read_excel(xlsx_path)
    print(f"  {os.path.basename(xlsx_path)}: {len(df)} rows, {len(df.columns)} columns")
    print(f"    Columns: {list(df.columns)}")
    print()